In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
%cd /content/drive/MyDrive/ECE232_Project2/

Mounted at /content/drive/
/content/drive/MyDrive/ECE232_Project2


# Q1

In [2]:
# ===== Mount Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

# ===== Import =====
import os
import networkx as nx
import matplotlib.pyplot as plt

# ===== Path =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")

OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== Graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Q1.1 =====
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
is_connected = nx.is_connected(G)

print("\n===== Q1.1 =====")
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)
print("Is connected:", is_connected)

# ===== Q1.2 GCC =====
if is_connected:
    gcc = G
else:
    gcc_nodes = max(nx.connected_components(G), key=len)
    gcc = G.subgraph(gcc_nodes).copy()

print("\n===== Q1.2 =====")
print("GCC size:", gcc.number_of_nodes())

# ===== Graph 1：Degree Distribution Histogram =====
print("\nPlotting degree histogram...")
degrees = [d for _, d in G.degree()]

plt.figure(figsize=(7,5))
plt.hist(degrees, bins=50)
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution (Histogram)")

save_path1 = os.path.join(OUTPUT_DIR, "q1_degree_histogram.png")
plt.savefig(save_path1, dpi=300)
plt.close()

print("Saved:", save_path1)

# ===== Graph 22：Network Visualization =====
print("\nPlotting network visualization (subset)...")
subset_nodes = list(G.nodes())[:200]
subG = G.subgraph(subset_nodes)

plt.figure(figsize=(8,8))
pos = nx.spring_layout(subG, seed=42)

nx.draw(subG, pos,
        node_size=20,
        edge_color="gray",
        with_labels=False)

save_path2 = os.path.join(OUTPUT_DIR, "q1_network_visualization.png")
plt.savefig(save_path2, dpi=300)
plt.close()

print("Saved:", save_path2)

print("\n===== DONE Q1 =====")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading graph...
Graph loaded.

===== Q1.1 =====
Number of nodes: 4039
Number of edges: 88234
Is connected: True

===== Q1.2 =====
GCC size: 4039

Plotting degree histogram...
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q1_degree_histogram.png

Plotting network visualization (subset)...
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q1_network_visualization.png

===== DONE Q1 =====


#Q2

In [3]:
import os
import networkx as nx

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Q2 =====
print("\n===== Q2 =====")

is_connected = nx.is_connected(G)
print("Is connected:", is_connected)

if is_connected:
    H = G
    graph_name = "Facebook network"
else:
    gcc_nodes = max(nx.connected_components(G), key=len)
    H = G.subgraph(gcc_nodes).copy()
    graph_name = "Giant Connected Component (GCC)"

print(f"Computing diameter on: {graph_name}")
diameter_value = nx.diameter(H, usebounds=True)

print(f"Diameter of {graph_name}: {diameter_value}")
print("\n===== DONE Q2 =====")

Loading graph...
Graph loaded.

===== Q2 =====
Is connected: True
Computing diameter on: Facebook network
Diameter of Facebook network: 8

===== DONE Q2 =====


#Q3+Q4

In [4]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Degree statistics =====
degrees = [d for _, d in G.degree()]
average_degree = sum(degrees) / len(degrees)

print("\n===== Q3 =====")
print("Average degree:", average_degree)

# ===== Q3: Degree distribution =====
degree_count = Counter(degrees)
x = sorted(degree_count.keys())
y = [degree_count[k] for k in x]

plt.figure(figsize=(7, 5))
plt.scatter(x, y, s=18)
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution of Facebook Network")
plt.tight_layout()

save_path_q3 = os.path.join(OUTPUT_DIR, "q3_degree_distribution.png")
plt.savefig(save_path_q3, dpi=300)
plt.close()

print("Saved:", save_path_q3)

# ===== Q4: Log-log degree distribution =====
x_all = np.array([k for k in x if k > 0 and degree_count[k] > 0])
y_all = np.array([degree_count[k] for k in x_all])

# Fit only the tail
k_min = 10
x_fit = np.array([k for k in x_all if k >= k_min])
y_fit = np.array([degree_count[k] for k in x_fit])

log_x_fit = np.log10(x_fit)
log_y_fit = np.log10(y_fit)

slope, intercept = np.polyfit(log_x_fit, log_y_fit, 1)
fitted_y = 10 ** (intercept + slope * np.log10(x_fit))

print("\n===== Q4 =====")
print("k_min:", k_min)
print("Estimated slope:", slope)
print("Estimated intercept:", intercept)

plt.figure(figsize=(7, 5))
plt.scatter(x_all, y_all, s=18, label="Data")
plt.plot(x_fit, fitted_y, linewidth=2, label=f"Tail fit: slope = {slope:.4f}, k_min={k_min}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Degree (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Log-Log Degree Distribution of Facebook Network")
plt.legend()
plt.tight_layout()

save_path_q4 = os.path.join(OUTPUT_DIR, "q4_loglog_degree_distribution_tail_fit.png")
plt.savefig(save_path_q4, dpi=300)
plt.close()

print("Saved:", save_path_q4)

print("\n===== DONE Q3 + Q4 =====")

Loading graph...
Graph loaded.

===== Q3 =====
Average degree: 43.69101262688784
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q3_degree_distribution.png

===== Q4 =====
k_min: 10
Estimated slope: -1.4436111716094018
Estimated intercept: 3.721006763655466
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q4_loglog_degree_distribution_tail_fit.png

===== DONE Q3 + Q4 =====


#Q5

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helper =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

# ===== Q5 =====
print("\n===== Q5 =====")

graph_node_id = 1
core_node = graph_id_to_edgelist_id(graph_node_id)

ego_nodes = [core_node] + list(G.neighbors(core_node))
P = G.subgraph(ego_nodes).copy()

print("Graph node ID:", graph_node_id)
print("Edgelist node ID:", core_node)
print("Number of nodes in personalized network:", P.number_of_nodes())
print("Number of edges in personalized network:", P.number_of_edges())

# ===== Optional visualization =====
plt.figure(figsize=(8, 8))
pos = nx.spring_layout(P, seed=42)

node_colors = ["black" if n == core_node else "lightgray" for n in P.nodes()]
node_sizes = [180 if n == core_node else 40 for n in P.nodes()]

nx.draw(
    P,
    pos,
    node_color=node_colors,
    node_size=node_sizes,
    edge_color="gray",
    with_labels=False
)

plt.title(f"Personalized Network of Node {graph_node_id}")
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "q5_personalized_network_node_1.png")
plt.savefig(save_path, dpi=300)
plt.close()

print("Saved:", save_path)
print("\n===== DONE Q5 =====")

Loading graph...
Graph loaded.

===== Q5 =====
Graph node ID: 1
Edgelist node ID: 0
Number of nodes in personalized network: 348
Number of edges in personalized network: 2866


/tmp/ipykernel_48568/1606867596.py:51: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q5_personalized_network_node_1.png

===== DONE Q5 =====


#Q6

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helper =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

# ===== Build personalized network of node ID 1 =====
graph_node_id = 1
core_node = graph_id_to_edgelist_id(graph_node_id)

ego_nodes = [core_node] + list(G.neighbors(core_node))
P = G.subgraph(ego_nodes).copy()

# ===== Q6 =====
print("\n===== Q6 =====")

is_connected = nx.is_connected(P)
print("Is the personalized network connected:", is_connected)

if is_connected:
    H = P
    graph_name = "personalized network"
else:
    gcc_nodes = max(nx.connected_components(P), key=len)
    H = P.subgraph(gcc_nodes).copy()
    graph_name = "GCC of the personalized network"

diameter_value = nx.diameter(H, usebounds=True)

lower_bound = 1
upper_bound = 2

print(f"Diameter of the {graph_name}: {diameter_value}")
print("Trivial lower bound:", lower_bound)
print("Trivial upper bound:", upper_bound)

# ===== Optional plot =====
lengths = nx.single_source_shortest_path_length(H, core_node)

plt.figure(figsize=(7, 5))
plt.hist(
    list(lengths.values()),
    bins=range(0, max(lengths.values()) + 2),
    align="left",
    rwidth=0.8
)
plt.xlabel("Shortest Path Length from Core Node")
plt.ylabel("Frequency")
plt.title(f"Shortest Path Distribution in Personalized Network of Node {graph_node_id}")
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, "q6_shortest_path_distribution_node_1.png")
plt.savefig(save_path, dpi=300)
plt.close()

print("Saved:", save_path)
print("\n===== DONE Q6 =====")

Loading graph...
Graph loaded.

===== Q6 =====
Is the personalized network connected: True
Diameter of the personalized network: 2
Trivial lower bound: 1
Trivial upper bound: 2
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q6_shortest_path_distribution_node_1.png

===== DONE Q6 =====


#Q8

In [ ]:
import os
import numpy as np
import networkx as nx

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Q8 =====
print("\n===== Q8 =====")

core_nodes = [node for node, degree in G.degree() if degree > 200]
core_degrees = [G.degree(node) for node in core_nodes]

num_core_nodes = len(core_nodes)
average_core_degree = np.mean(core_degrees)

print("Number of core nodes:", num_core_nodes)
print("Average degree of core nodes:", average_core_degree)

print("\nCore nodes:")
print(core_nodes)

print("\n===== DONE Q8 =====")

Loading graph...
Graph loaded.

===== Q8 =====
Number of core nodes: 40
Average degree of core nodes: 279.375

Core nodes:
[0, 107, 348, 1684, 1912, 483, 1086, 1199, 1352, 1431, 1584, 1589, 1663, 1730, 1746, 1768, 1800, 1827, 1888, 2543, 3437, 1941, 2047, 2266, 2347, 1985, 1993, 2078, 2123, 2142, 2206, 2218, 2229, 2233, 2240, 2410, 2464, 2507, 2560, 2611]

===== DONE Q8 =====


#Q9

In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

try:
    import igraph as ig
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-igraph", "-q"])
    import igraph as ig

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q9_DIR = os.path.join(OUTPUT_DIR, "q9")
os.makedirs(Q9_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def build_personalized_network(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    return G.subgraph(ego_nodes).copy(), core_node

def nx_to_igraph_with_names(nx_graph):
    nodes = list(nx_graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    edges = [(node_to_index[u], node_to_index[v]) for u, v in nx_graph.edges()]
    ig_graph = ig.Graph(edges=edges, directed=False)
    ig_graph.vs["name"] = [str(node) for node in nodes]
    return ig_graph, nodes

def detect_communities(ig_graph, method):
    if method == "fast_greedy":
        return ig_graph.community_fastgreedy().as_clustering()
    elif method == "edge_betweenness":
        return ig_graph.community_edge_betweenness().as_clustering()
    elif method == "infomap":
        return ig_graph.community_infomap()
    else:
        raise ValueError("Unknown method")

def plot_communities(nx_graph, core_node, membership, node_order, title, save_path):
    community_by_node = {
        node_order[i]: membership[i]
        for i in range(len(node_order))
    }

    node_colors = [community_by_node[node] for node in nx_graph.nodes()]
    node_sizes = [220 if node == core_node else 45 for node in nx_graph.nodes()]

    plt.figure(figsize=(9, 9))
    pos = nx.spring_layout(nx_graph, seed=42)

    nx.draw_networkx_edges(
        nx_graph,
        pos,
        alpha=0.25,
        width=0.6
    )

    nx.draw_networkx_nodes(
        nx_graph,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=plt.cm.tab20
    )

    nx.draw_networkx_nodes(
        nx_graph,
        pos,
        nodelist=[core_node],
        node_color="black",
        node_size=260
    )

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# ===== Q9 =====
print("\n===== Q9 =====")

target_graph_node_ids = [1, 108, 349, 484, 1087]
methods = ["fast_greedy", "edge_betweenness", "infomap"]

results = []

for graph_node_id in target_graph_node_ids:
    P, core_node = build_personalized_network(G, graph_node_id)
    ig_graph, node_order = nx_to_igraph_with_names(P)

    print(f"\nNode ID {graph_node_id}")
    print("Personalized network nodes:", P.number_of_nodes())
    print("Personalized network edges:", P.number_of_edges())

    for method in methods:
        clustering = detect_communities(ig_graph, method)
        modularity = ig_graph.modularity(clustering.membership)
        num_communities = len(set(clustering.membership))

        results.append({
            "graph_node_id": graph_node_id,
            "edgelist_node_id": core_node,
            "method": method,
            "num_nodes": P.number_of_nodes(),
            "num_edges": P.number_of_edges(),
            "num_communities": num_communities,
            "modularity": modularity
        })

        print(f"{method}: communities = {num_communities}, modularity = {modularity:.6f}")

        save_path = os.path.join(
            Q9_DIR,
            f"q9_node_{graph_node_id}_{method}.png"
        )

        title = f"Node {graph_node_id} Personalized Network - {method}"
        plot_communities(
            nx_graph=P,
            core_node=core_node,
            membership=clustering.membership,
            node_order=node_order,
            title=title,
            save_path=save_path
        )

        print("Saved:", save_path)

# ===== Save modularity table =====
results_df = pd.DataFrame(results)
csv_path = os.path.join(Q9_DIR, "q9_modularity_scores.csv")
results_df.to_csv(csv_path, index=False)

print("\nSaved modularity table:", csv_path)
print("\n===== Q9 Results =====")
print(results_df)
print("\n===== DONE Q9 =====")

Loading graph...
Graph loaded.

===== Q9 =====

Node ID 1
Personalized network nodes: 348
Personalized network edges: 2866
fast_greedy: communities = 8, modularity = 0.413101
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q9/q9_node_1_fast_greedy.png
edge_betweenness: communities = 41, modularity = 0.353302
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q9/q9_node_1_edge_betweenness.png
infomap: communities = 24, modularity = 0.388689
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q9/q9_node_1_infomap.png

Node ID 108
Personalized network nodes: 1046
Personalized network edges: 27795
fast_greedy: communities = 9, modularity = 0.435929
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q9/q9_node_108_fast_greedy.png
edge_betweenness: communities = 52, modularity = 0.506755
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q9/q9_node_108_edge_betweenness.png
infomap: communities = 23, modularity = 0.508985
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/

#Q10

In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

try:
    import igraph as ig
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-igraph", "-q"])
    import igraph as ig

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q10_DIR = os.path.join(OUTPUT_DIR, "q10")
os.makedirs(Q10_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def build_personalized_network_without_core(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    P = G.subgraph(ego_nodes).copy()
    P.remove_node(core_node)
    return P, core_node

def nx_to_igraph_with_names(nx_graph):
    nodes = list(nx_graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    edges = [(node_to_index[u], node_to_index[v]) for u, v in nx_graph.edges()]
    ig_graph = ig.Graph(edges=edges, directed=False)
    ig_graph.vs["name"] = [str(node) for node in nodes]
    return ig_graph, nodes

def detect_communities(ig_graph, method):
    if method == "fast_greedy":
        return ig_graph.community_fastgreedy().as_clustering()
    elif method == "edge_betweenness":
        return ig_graph.community_edge_betweenness().as_clustering()
    elif method == "infomap":
        return ig_graph.community_infomap()
    else:
        raise ValueError("Unknown method")

def plot_communities(nx_graph, membership, node_order, title, save_path):
    community_by_node = {
        node_order[i]: membership[i]
        for i in range(len(node_order))
    }

    node_colors = [community_by_node[node] for node in nx_graph.nodes()]

    plt.figure(figsize=(9, 9))
    pos = nx.spring_layout(nx_graph, seed=42)

    nx.draw_networkx_edges(
        nx_graph,
        pos,
        alpha=0.25,
        width=0.6
    )

    nx.draw_networkx_nodes(
        nx_graph,
        pos,
        node_color=node_colors,
        node_size=45,
        cmap=plt.cm.tab20
    )

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# ===== Q10 =====
print("\n===== Q10 =====")

target_graph_node_ids = [1, 108, 349, 484, 1087]
methods = ["fast_greedy", "edge_betweenness", "infomap"]

results = []

for graph_node_id in target_graph_node_ids:
    P_removed, core_node = build_personalized_network_without_core(G, graph_node_id)
    ig_graph, node_order = nx_to_igraph_with_names(P_removed)

    print(f"\nNode ID {graph_node_id}")
    print("Removed core edgelist node ID:", core_node)
    print("Modified personalized network nodes:", P_removed.number_of_nodes())
    print("Modified personalized network edges:", P_removed.number_of_edges())

    for method in methods:
        clustering = detect_communities(ig_graph, method)
        modularity = ig_graph.modularity(clustering.membership)
        num_communities = len(set(clustering.membership))

        results.append({
            "graph_node_id": graph_node_id,
            "removed_edgelist_node_id": core_node,
            "method": method,
            "num_nodes": P_removed.number_of_nodes(),
            "num_edges": P_removed.number_of_edges(),
            "num_communities": num_communities,
            "modularity": modularity
        })

        print(f"{method}: communities = {num_communities}, modularity = {modularity:.6f}")

        save_path = os.path.join(
            Q10_DIR,
            f"q10_node_{graph_node_id}_removed_core_{method}.png"
        )

        title = f"Node {graph_node_id} Personalized Network without Core - {method}"
        plot_communities(
            nx_graph=P_removed,
            membership=clustering.membership,
            node_order=node_order,
            title=title,
            save_path=save_path
        )

        print("Saved:", save_path)

# ===== Save modularity table =====
results_df = pd.DataFrame(results)
csv_path = os.path.join(Q10_DIR, "q10_modularity_scores.csv")
results_df.to_csv(csv_path, index=False)

print("\nSaved modularity table:", csv_path)
print("\n===== Q10 Results =====")
print(results_df)
print("\n===== DONE Q10 =====")

Loading graph...
Graph loaded.

===== Q10 =====

Node ID 1
Removed core edgelist node ID: 0
Modified personalized network nodes: 347
Modified personalized network edges: 2519
fast_greedy: communities = 26, modularity = 0.441853
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q10/q10_node_1_removed_core_fast_greedy.png
edge_betweenness: communities = 50, modularity = 0.416146
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q10/q10_node_1_removed_core_edge_betweenness.png
infomap: communities = 36, modularity = 0.419217
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q10/q10_node_1_removed_core_infomap.png

Node ID 108
Removed core edgelist node ID: 107
Modified personalized network nodes: 1045
Modified personalized network edges: 26750
fast_greedy: communities = 22, modularity = 0.458127
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q10/q10_node_108_removed_core_fast_greedy.png
edge_betweenness: communities = 57, modularity = 0.521322
Saved: /content/drive/MyDr

#Q12

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q12_DIR = os.path.join(OUTPUT_DIR, "q12")
os.makedirs(Q12_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def build_personalized_network(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    P = G.subgraph(ego_nodes).copy()
    return P, core_node

def compute_embeddedness(P, core_node, node):
    core_neighbors = set(P.neighbors(core_node))
    node_neighbors = set(P.neighbors(node))
    return len(core_neighbors.intersection(node_neighbors))

def compute_dispersion(P, core_node, node):
    mutual_friends = list(
        set(P.neighbors(core_node)).intersection(set(P.neighbors(node)))
    )

    if len(mutual_friends) < 2:
        return 0

    H = P.copy()
    H.remove_node(core_node)
    H.remove_node(node)

    dispersion_value = 0

    for u, v in combinations(mutual_friends, 2):
        try:
            dispersion_value += nx.shortest_path_length(H, u, v)
        except nx.NetworkXNoPath:
            continue

    return dispersion_value

# ===== Q12 =====
print("\n===== Q12 =====")

target_graph_node_ids = [1, 108, 349, 484, 1087]

summary_rows = []

for graph_node_id in target_graph_node_ids:
    P, core_node = build_personalized_network(G, graph_node_id)

    embeddedness_values = []
    dispersion_values = []

    print(f"\nProcessing node ID {graph_node_id}")
    print("Personalized network nodes:", P.number_of_nodes())
    print("Personalized network edges:", P.number_of_edges())

    for node in P.nodes():
        if node == core_node:
            continue

        emb = compute_embeddedness(P, core_node, node)
        disp = compute_dispersion(P, core_node, node)

        embeddedness_values.append(emb)
        dispersion_values.append(disp)

    summary_rows.append({
        "graph_node_id": graph_node_id,
        "num_non_core_nodes": len(embeddedness_values),
        "max_embeddedness": max(embeddedness_values),
        "max_dispersion": max(dispersion_values),
        "avg_embeddedness": sum(embeddedness_values) / len(embeddedness_values),
        "avg_dispersion": sum(dispersion_values) / len(dispersion_values)
    })

    plt.figure(figsize=(7, 5))
    plt.hist(embeddedness_values, bins=30)
    plt.xlabel("Embeddedness")
    plt.ylabel("Frequency")
    plt.title(f"Embeddedness Distribution for Node {graph_node_id}")
    plt.tight_layout()

    emb_path = os.path.join(Q12_DIR, f"q12_node_{graph_node_id}_embeddedness_hist.png")
    plt.savefig(emb_path, dpi=300)
    plt.close()

    print("Saved:", emb_path)

    plt.figure(figsize=(7, 5))
    plt.hist(dispersion_values, bins=30)
    plt.xlabel("Dispersion")
    plt.ylabel("Frequency")
    plt.title(f"Dispersion Distribution for Node {graph_node_id}")
    plt.tight_layout()

    disp_path = os.path.join(Q12_DIR, f"q12_node_{graph_node_id}_dispersion_hist.png")
    plt.savefig(disp_path, dpi=300)
    plt.close()

    print("Saved:", disp_path)

print("\n===== Q12 Summary =====")
for row in summary_rows:
    print(row)

print("\n===== DONE Q12 =====")

Loading graph...
Graph loaded.

===== Q12 =====

Processing node ID 1
Personalized network nodes: 348
Personalized network edges: 2866
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_1_embeddedness_hist.png
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_1_dispersion_hist.png

Processing node ID 108
Personalized network nodes: 1046
Personalized network edges: 27795
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_108_embeddedness_hist.png
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_108_dispersion_hist.png

Processing node ID 349
Personalized network nodes: 230
Personalized network edges: 3441
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_349_embeddedness_hist.png
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q12/q12_node_349_dispersion_hist.png

Processing node ID 484
Personalized network nodes: 232
Personalized network edges: 4525
Saved: /content/drive/MyDrive/ECE232_Project2/ou

#Q13

In [ ]:
import os
import sys
import subprocess
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

try:
    import igraph as ig
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-igraph", "-q"])
    import igraph as ig

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q13_DIR = os.path.join(OUTPUT_DIR, "q13")
os.makedirs(Q13_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def edgelist_id_to_graph_id(edgelist_node_id):
    return edgelist_node_id + 1

def build_personalized_network(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    P = G.subgraph(ego_nodes).copy()
    return P, core_node

def compute_dispersion(P, core_node, node):
    mutual_friends = list(
        set(P.neighbors(core_node)).intersection(set(P.neighbors(node)))
    )

    if len(mutual_friends) < 2:
        return 0

    H = P.copy()
    H.remove_node(core_node)
    H.remove_node(node)

    dispersion_value = 0

    for u, v in combinations(mutual_friends, 2):
        try:
            dispersion_value += nx.shortest_path_length(H, u, v)
        except nx.NetworkXNoPath:
            continue

    return dispersion_value

def nx_to_igraph_with_names(nx_graph):
    nodes = list(nx_graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    edges = [(node_to_index[u], node_to_index[v]) for u, v in nx_graph.edges()]
    ig_graph = ig.Graph(edges=edges, directed=False)
    ig_graph.vs["name"] = [str(node) for node in nodes]
    return ig_graph, nodes

def get_fast_greedy_membership(nx_graph):
    ig_graph, node_order = nx_to_igraph_with_names(nx_graph)
    clustering = ig_graph.community_fastgreedy().as_clustering()

    membership_by_node = {
        node_order[i]: clustering.membership[i]
        for i in range(len(node_order))
    }

    modularity = ig_graph.modularity(clustering.membership)
    return membership_by_node, modularity

def plot_q13(P, core_node, max_disp_node, membership_by_node, title, save_path):
    pos = nx.spring_layout(P, seed=42)

    node_colors = [membership_by_node[node] for node in P.nodes()]
    node_sizes = []
    node_edgecolors = []

    for node in P.nodes():
        if node == core_node:
            node_sizes.append(260)
            node_edgecolors.append("black")
        elif node == max_disp_node:
            node_sizes.append(260)
            node_edgecolors.append("red")
        else:
            node_sizes.append(45)
            node_edgecolors.append("black")

    incident_edges = set(P.edges(max_disp_node))
    incident_edges = set(tuple(sorted(edge)) for edge in incident_edges)
    normal_edges = []
    highlighted_edges = []

    for edge in P.edges():
        sorted_edge = tuple(sorted(edge))
        if sorted_edge in incident_edges:
            highlighted_edges.append(edge)
        else:
            normal_edges.append(edge)

    plt.figure(figsize=(9, 9))

    nx.draw_networkx_edges(
        P,
        pos,
        edgelist=normal_edges,
        alpha=0.18,
        width=0.5
    )

    nx.draw_networkx_edges(
        P,
        pos,
        edgelist=highlighted_edges,
        edge_color="red",
        alpha=0.9,
        width=1.5
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        edgecolors=node_edgecolors,
        linewidths=0.8,
        cmap=plt.cm.tab20
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        nodelist=[core_node],
        node_color="black",
        node_size=280
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        nodelist=[max_disp_node],
        node_color="red",
        node_size=280
    )

    labels = {
        core_node: f"core {edgelist_id_to_graph_id(core_node)}",
        max_disp_node: f"max disp {edgelist_id_to_graph_id(max_disp_node)}"
    }

    nx.draw_networkx_labels(
        P,
        pos,
        labels=labels,
        font_size=8,
        font_color="black"
    )

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# ===== Q13 =====
print("\n===== Q13 =====")

target_graph_node_ids = [1, 108, 349, 484, 1087]
summary_rows = []

for graph_node_id in target_graph_node_ids:
    print(f"\nProcessing node ID {graph_node_id}")

    P, core_node = build_personalized_network(G, graph_node_id)

    dispersion_by_node = {}
    for node in P.nodes():
        if node == core_node:
            continue
        dispersion_by_node[node] = compute_dispersion(P, core_node, node)

    max_disp_node = max(dispersion_by_node, key=dispersion_by_node.get)
    max_disp_value = dispersion_by_node[max_disp_node]

    membership_by_node, modularity = get_fast_greedy_membership(P)

    save_path = os.path.join(Q13_DIR, f"q13_node_{graph_node_id}_max_dispersion.png")

    title = (
        f"Node {graph_node_id} Personalized Network\n"
        f"Fast-Greedy Communities, Max Dispersion Node = {edgelist_id_to_graph_id(max_disp_node)}"
    )

    plot_q13(
        P=P,
        core_node=core_node,
        max_disp_node=max_disp_node,
        membership_by_node=membership_by_node,
        title=title,
        save_path=save_path
    )

    summary_rows.append({
        "graph_node_id": graph_node_id,
        "max_dispersion_edgelist_node_id": max_disp_node,
        "max_dispersion_graph_node_id": edgelist_id_to_graph_id(max_disp_node),
        "max_dispersion_value": max_disp_value,
        "fast_greedy_modularity": modularity,
        "saved_plot": save_path
    })

    print("Max dispersion edgelist node ID:", max_disp_node)
    print("Max dispersion graph node ID:", edgelist_id_to_graph_id(max_disp_node))
    print("Max dispersion value:", max_disp_value)
    print("Fast-Greedy modularity:", modularity)
    print("Saved:", save_path)

print("\n===== Q13 Summary =====")
for row in summary_rows:
    print(row)

print("\n===== DONE Q13 =====")

Loading graph...
Graph loaded.

===== Q13 =====

Processing node ID 1
Max dispersion edgelist node ID: 56
Max dispersion graph node ID: 57
Max dispersion value: 4971
Fast-Greedy modularity: 0.41310137283423476
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q13/q13_node_1_max_dispersion.png

Processing node ID 108
Max dispersion edgelist node ID: 1888
Max dispersion graph node ID: 1889
Max dispersion value: 51247
Fast-Greedy modularity: 0.435929376026475
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q13/q13_node_108_max_dispersion.png

Processing node ID 349
Max dispersion edgelist node ID: 376
Max dispersion graph node ID: 377
Max dispersion value: 8258
Fast-Greedy modularity: 0.2503460796905125
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q13/q13_node_349_max_dispersion.png

Processing node ID 484
Max dispersion edgelist node ID: 107
Max dispersion graph node ID: 108
Max dispersion value: 29837
Fast-Greedy modularity: 0.5070016421965142
Saved: /content/drive/My

#Q14

In [ ]:
import os
import sys
import subprocess
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations

try:
    import igraph as ig
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-igraph", "-q"])
    import igraph as ig

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q14_DIR = os.path.join(OUTPUT_DIR, "q14")
os.makedirs(Q14_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def edgelist_id_to_graph_id(edgelist_node_id):
    return edgelist_node_id + 1

def build_personalized_network(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    P = G.subgraph(ego_nodes).copy()
    return P, core_node

def compute_embeddedness(P, core_node, node):
    core_neighbors = set(P.neighbors(core_node))
    node_neighbors = set(P.neighbors(node))
    return len(core_neighbors.intersection(node_neighbors))

def compute_dispersion(P, core_node, node):
    mutual_friends = list(
        set(P.neighbors(core_node)).intersection(set(P.neighbors(node)))
    )

    if len(mutual_friends) < 2:
        return 0

    H = P.copy()
    H.remove_node(core_node)
    H.remove_node(node)

    dispersion_value = 0

    for u, v in combinations(mutual_friends, 2):
        try:
            dispersion_value += nx.shortest_path_length(H, u, v)
        except nx.NetworkXNoPath:
            continue

    return dispersion_value

def nx_to_igraph_with_names(nx_graph):
    nodes = list(nx_graph.nodes())
    node_to_index = {node: idx for idx, node in enumerate(nodes)}
    edges = [(node_to_index[u], node_to_index[v]) for u, v in nx_graph.edges()]
    ig_graph = ig.Graph(edges=edges, directed=False)
    ig_graph.vs["name"] = [str(node) for node in nodes]
    return ig_graph, nodes

def get_fast_greedy_membership(nx_graph):
    ig_graph, node_order = nx_to_igraph_with_names(nx_graph)
    clustering = ig_graph.community_fastgreedy().as_clustering()

    membership_by_node = {
        node_order[i]: clustering.membership[i]
        for i in range(len(node_order))
    }

    modularity = ig_graph.modularity(clustering.membership)
    return membership_by_node, modularity

def plot_q14(P, core_node, max_emb_node, max_ratio_node, membership_by_node, title, save_path):
    pos = nx.spring_layout(P, seed=42)

    node_colors = [membership_by_node[node] for node in P.nodes()]
    node_sizes = []
    node_edgecolors = []

    for node in P.nodes():
        if node == core_node:
            node_sizes.append(280)
            node_edgecolors.append("black")
        elif node == max_emb_node:
            node_sizes.append(300)
            node_edgecolors.append("blue")
        elif node == max_ratio_node:
            node_sizes.append(300)
            node_edgecolors.append("red")
        else:
            node_sizes.append(45)
            node_edgecolors.append("black")

    highlighted_edges_emb = set(tuple(sorted(edge)) for edge in P.edges(max_emb_node))
    highlighted_edges_ratio = set(tuple(sorted(edge)) for edge in P.edges(max_ratio_node))

    normal_edges = []
    emb_edges = []
    ratio_edges = []

    for edge in P.edges():
        sorted_edge = tuple(sorted(edge))
        if sorted_edge in highlighted_edges_emb:
            emb_edges.append(edge)
        elif sorted_edge in highlighted_edges_ratio:
            ratio_edges.append(edge)
        else:
            normal_edges.append(edge)

    plt.figure(figsize=(9, 9))

    nx.draw_networkx_edges(
        P,
        pos,
        edgelist=normal_edges,
        alpha=0.16,
        width=0.5
    )

    nx.draw_networkx_edges(
        P,
        pos,
        edgelist=emb_edges,
        edge_color="blue",
        alpha=0.8,
        width=1.3
    )

    nx.draw_networkx_edges(
        P,
        pos,
        edgelist=ratio_edges,
        edge_color="red",
        alpha=0.8,
        width=1.3
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        edgecolors=node_edgecolors,
        linewidths=0.9,
        cmap=plt.cm.tab20
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        nodelist=[core_node],
        node_color="black",
        node_size=300
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        nodelist=[max_emb_node],
        node_color="blue",
        node_size=320
    )

    nx.draw_networkx_nodes(
        P,
        pos,
        nodelist=[max_ratio_node],
        node_color="red",
        node_size=320
    )

    labels = {
        core_node: f"core {edgelist_id_to_graph_id(core_node)}",
        max_emb_node: f"max emb {edgelist_id_to_graph_id(max_emb_node)}",
        max_ratio_node: f"max disp/emb {edgelist_id_to_graph_id(max_ratio_node)}"
    }

    nx.draw_networkx_labels(
        P,
        pos,
        labels=labels,
        font_size=8,
        font_color="black"
    )

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

# ===== Q14 =====
print("\n===== Q14 =====")

target_graph_node_ids = [1, 108, 349, 484, 1087]
summary_rows = []

for graph_node_id in target_graph_node_ids:
    print(f"\nProcessing node ID {graph_node_id}")

    P, core_node = build_personalized_network(G, graph_node_id)

    embeddedness_by_node = {}
    dispersion_by_node = {}
    ratio_by_node = {}

    for node in P.nodes():
        if node == core_node:
            continue

        emb = compute_embeddedness(P, core_node, node)
        disp = compute_dispersion(P, core_node, node)

        embeddedness_by_node[node] = emb
        dispersion_by_node[node] = disp

        if emb > 0:
            ratio_by_node[node] = disp / emb

    max_emb_node = max(embeddedness_by_node, key=embeddedness_by_node.get)
    max_ratio_node = max(ratio_by_node, key=ratio_by_node.get)

    max_emb_value = embeddedness_by_node[max_emb_node]
    max_ratio_value = ratio_by_node[max_ratio_node]
    max_ratio_disp = dispersion_by_node[max_ratio_node]
    max_ratio_emb = embeddedness_by_node[max_ratio_node]

    membership_by_node, modularity = get_fast_greedy_membership(P)

    save_path = os.path.join(Q14_DIR, f"q14_node_{graph_node_id}_max_emb_and_ratio.png")

    title = (
        f"Node {graph_node_id} Personalized Network\n"
        f"Blue: Max Embeddedness, Red: Max Dispersion/Embeddedness"
    )

    plot_q14(
        P=P,
        core_node=core_node,
        max_emb_node=max_emb_node,
        max_ratio_node=max_ratio_node,
        membership_by_node=membership_by_node,
        title=title,
        save_path=save_path
    )

    summary_rows.append({
        "graph_node_id": graph_node_id,
        "max_emb_edgelist_node_id": max_emb_node,
        "max_emb_graph_node_id": edgelist_id_to_graph_id(max_emb_node),
        "max_embeddedness": max_emb_value,
        "max_ratio_edgelist_node_id": max_ratio_node,
        "max_ratio_graph_node_id": edgelist_id_to_graph_id(max_ratio_node),
        "max_ratio_dispersion": max_ratio_disp,
        "max_ratio_embeddedness": max_ratio_emb,
        "max_dispersion_over_embeddedness": max_ratio_value,
        "fast_greedy_modularity": modularity,
        "saved_plot": save_path
    })

    print("Max embeddedness edgelist node ID:", max_emb_node)
    print("Max embeddedness graph node ID:", edgelist_id_to_graph_id(max_emb_node))
    print("Max embeddedness value:", max_emb_value)

    print("Max dispersion/embeddedness edgelist node ID:", max_ratio_node)
    print("Max dispersion/embeddedness graph node ID:", edgelist_id_to_graph_id(max_ratio_node))
    print("Dispersion:", max_ratio_disp)
    print("Embeddedness:", max_ratio_emb)
    print("Dispersion/Embeddedness:", max_ratio_value)

    print("Fast-Greedy modularity:", modularity)
    print("Saved:", save_path)

print("\n===== Q14 Summary =====")
for row in summary_rows:
    print(row)

print("\n===== DONE Q14 =====")

Loading graph...
Graph loaded.

===== Q14 =====

Processing node ID 1
Max embeddedness edgelist node ID: 56
Max embeddedness graph node ID: 57
Max embeddedness value: 77
Max dispersion/embeddedness edgelist node ID: 25
Max dispersion/embeddedness graph node ID: 26
Dispersion: 4503
Embeddedness: 68
Dispersion/Embeddedness: 66.22058823529412
Fast-Greedy modularity: 0.41310137283423476
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q14/q14_node_1_max_emb_and_ratio.png

Processing node ID 108
Max embeddedness edgelist node ID: 1888
Max embeddedness graph node ID: 1889
Max embeddedness value: 253
Max dispersion/embeddedness edgelist node ID: 1888
Max dispersion/embeddedness graph node ID: 1889
Dispersion: 51247
Embeddedness: 253
Dispersion/Embeddedness: 202.55731225296444
Fast-Greedy modularity: 0.435929376026475
Saved: /content/drive/MyDrive/ECE232_Project2/outputs/q14/q14_node_108_max_emb_and_ratio.png

Processing node ID 349
Max embeddedness edgelist node ID: 376
Max embeddedness 

#Q16

In [ ]:
import os
import networkx as nx

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helper =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

# ===== Q16 =====
print("\n===== Q16 =====")

graph_node_id = 415
core_node = graph_id_to_edgelist_id(graph_node_id)

ego_nodes = [core_node] + list(G.neighbors(core_node))
P = G.subgraph(ego_nodes).copy()

Nr = [node for node, deg in P.degree() if deg == 24]

print("Graph node ID:", graph_node_id)
print("Edgelist node ID:", core_node)
print("|Nr| =", len(Nr))

print("\nNodes in Nr (graph node ID):")
print([n + 1 for n in Nr])

print("\n===== DONE Q16 =====")

Loading graph...
Graph loaded.

===== Q16 =====
Graph node ID: 415
Edgelist node ID: 414
|Nr| = 11

Nodes in Nr (graph node ID):
[579, 601, 616, 619, 628, 644, 659, 660, 662, 663, 497]

===== DONE Q16 =====


#Q17

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import networkx as nx

# ===== Reproducibility =====
random.seed(42)
np.random.seed(42)

# ===== Paths =====
DATA_DIR = "/content/drive/MyDrive/ECE232_Project2"
facebook_path = os.path.join(DATA_DIR, "facebook_combined.txt")
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
Q17_DIR = os.path.join(OUTPUT_DIR, "q17")
os.makedirs(Q17_DIR, exist_ok=True)

# ===== Load graph =====
print("Loading graph...")
G = nx.read_edgelist(facebook_path, nodetype=int)
print("Graph loaded.")

# ===== Helpers =====
def graph_id_to_edgelist_id(graph_node_id):
    return graph_node_id - 1

def edgelist_id_to_graph_id(edgelist_node_id):
    return edgelist_node_id + 1

def build_personalized_network(G, graph_node_id):
    core_node = graph_id_to_edgelist_id(graph_node_id)
    ego_nodes = [core_node] + list(G.neighbors(core_node))
    P = G.subgraph(ego_nodes).copy()
    return P, core_node

def common_neighbors_score(G, u, v):
    return len(set(G.neighbors(u)).intersection(set(G.neighbors(v))))

def jaccard_score(G, u, v):
    Nu = set(G.neighbors(u))
    Nv = set(G.neighbors(v))
    union_size = len(Nu.union(Nv))
    if union_size == 0:
        return 0.0
    return len(Nu.intersection(Nv)) / union_size

def adamic_adar_score(G, u, v):
    common = set(G.neighbors(u)).intersection(set(G.neighbors(v)))
    score = 0.0

    for w in common:
        degree_w = G.degree(w)
        if degree_w > 1:
            score += 1.0 / math.log(degree_w)

    return score

def score_pair(G, u, v, method):
    if method == "common_neighbors":
        return common_neighbors_score(G, u, v)
    elif method == "jaccard":
        return jaccard_score(G, u, v)
    elif method == "adamic_adar":
        return adamic_adar_score(G, u, v)
    else:
        raise ValueError("Unknown method")

def recommend_top_k(G_train, user, k, method):
    current_neighbors = set(G_train.neighbors(user))
    candidates = [
        node for node in G_train.nodes()
        if node != user and node not in current_neighbors
    ]

    scored_candidates = []
    for candidate in candidates:
        score = score_pair(G_train, user, candidate, method)
        scored_candidates.append((candidate, score))

    scored_candidates.sort(key=lambda x: x[1], reverse=True)
    return [node for node, score in scored_candidates[:k]]

def evaluate_user_accuracy(P_original, user, method, num_trials=10, remove_prob=0.25):
    original_neighbors = list(P_original.neighbors(user))
    trial_accuracies = []

    for trial in range(num_trials):
        removed_neighbors = [
            neighbor for neighbor in original_neighbors
            if random.random() < remove_prob
        ]

        if len(removed_neighbors) == 0:
            continue

        P_train = P_original.copy()
        P_train.remove_edges_from([(user, neighbor) for neighbor in removed_neighbors])

        predicted_neighbors = recommend_top_k(
            G_train=P_train,
            user=user,
            k=len(removed_neighbors),
            method=method
        )

        accuracy = len(set(predicted_neighbors).intersection(set(removed_neighbors))) / len(removed_neighbors)
        trial_accuracies.append(accuracy)

    if len(trial_accuracies) == 0:
        return 0.0

    return float(np.mean(trial_accuracies))

# ===== Q17 =====
print("\n===== Q17 =====")

P415, core_node = build_personalized_network(G, 415)

Nr = [node for node, degree in P415.degree() if degree == 24]

print("Personalized network of node ID 415")
print("Nodes:", P415.number_of_nodes())
print("Edges:", P415.number_of_edges())
print("|Nr|:", len(Nr))

methods = ["common_neighbors", "jaccard", "adamic_adar"]

user_level_rows = []
method_summary_rows = []

for method in methods:
    print(f"\nEvaluating method: {method}")

    user_accuracies = []

    for idx, user in enumerate(Nr):
        acc = evaluate_user_accuracy(
            P_original=P415,
            user=user,
            method=method,
            num_trials=10,
            remove_prob=0.25
        )

        user_accuracies.append(acc)

        user_level_rows.append({
            "method": method,
            "user_edgelist_node_id": user,
            "user_graph_node_id": edgelist_id_to_graph_id(user),
            "average_accuracy": acc
        })

        if (idx + 1) % 10 == 0 or (idx + 1) == len(Nr):
            print(f"Processed {idx + 1}/{len(Nr)} users")

    overall_accuracy = float(np.mean(user_accuracies))

    method_summary_rows.append({
        "method": method,
        "average_accuracy": overall_accuracy
    })

    print(f"Average accuracy for {method}: {overall_accuracy:.6f}")

# ===== Save results =====
user_level_df = pd.DataFrame(user_level_rows)
summary_df = pd.DataFrame(method_summary_rows)

user_level_path = os.path.join(Q17_DIR, "q17_user_level_accuracies.csv")
summary_path = os.path.join(Q17_DIR, "q17_method_summary.csv")

user_level_df.to_csv(user_level_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("\nSaved user-level results:", user_level_path)
print("Saved method summary:", summary_path)

print("\n===== Q17 Summary =====")
print(summary_df)

best_row = summary_df.loc[summary_df["average_accuracy"].idxmax()]
print("\nBest method:", best_row["method"])
print("Best average accuracy:", best_row["average_accuracy"])

print("\n===== DONE Q17 =====")

Loading graph...
Graph loaded.

===== Q17 =====
Personalized network of node ID 415
Nodes: 160
Edges: 1857
|Nr|: 11

Evaluating method: common_neighbors
Processed 10/11 users
Processed 11/11 users
Average accuracy for common_neighbors: 0.819001

Evaluating method: jaccard
Processed 10/11 users
Processed 11/11 users
Average accuracy for jaccard: 0.793235

Evaluating method: adamic_adar
Processed 10/11 users
Processed 11/11 users
Average accuracy for adamic_adar: 0.822269

Saved user-level results: /content/drive/MyDrive/ECE232_Project2/outputs/q17/q17_user_level_accuracies.csv
Saved method summary: /content/drive/MyDrive/ECE232_Project2/outputs/q17/q17_method_summary.csv

===== Q17 Summary =====
             method  average_accuracy
0  common_neighbors          0.819001
1           jaccard          0.793235
2       adamic_adar          0.822269

Best method: adamic_adar
Best average accuracy: 0.8222687918142463

===== DONE Q17 =====
